# Parcel-Level Voxel-Pattern ISC — Left-Wing Subjects

**Goal**: Measure inter-subject pattern correlation (ISPC) *within each parcel* of the
Schaefer 400 + Tian S3 atlas (432 parcels), for each of the four political-content
conditions, using left-wing subjects only.

**Analysis logic** (Chen et al., 2017, *Nature Neuroscience*, doi:10.1038/nn.4450):

1. For each subject and condition, load the pre-extracted NPZ `(n_posts, n_brain_voxels)`
2. Average across posts → one spatial pattern `(n_brain_voxels,)` per subject
3. For each parcel: slice the voxels belonging to that parcel → `(n_subjects, n_voxels_in_parcel)`
4. Leave-one-out ISC: for each subject i, compute Pearson r between their parcel pattern
   and the mean of all others — **correlation is across voxels** (spatial similarity)
5. Average r across subjects → one ISC value per parcel × condition (432 × 4 values)

**Key difference from `parcellated_leftwing_ispc.ipynb`**:  
That notebook correlates parcel activation *across posts* (timecourse covariance) — one
scalar per parcel per post.  
This notebook correlates voxel activation *across space within each parcel* (pattern
similarity) — one ISC value per parcel.

**Key difference from `roi_ispc_leftwing.ipynb`**:  
That notebook computes one ISC per ROI (2 values per condition).  
This notebook computes one ISC per parcel (432 values per condition), providing
whole-brain spatial coverage.

**Atlas**: Schaefer 2018 400 Parcels (7 Networks) + Tian S3 subcortical (32 regions)  
**Conditions**: AntiLeft · AntiRight · ProLeft · ProRight  
**Permutation test**: deferred — to be agreed with thesis supervisor

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys

sys.path.insert(0, str(Path.cwd().parent / 'src'))

from yy_fmri_kit.event_isc.parcel_pattern_similarity import (
    run_parcel_isc,
    results_to_dataframe,
)
from yy_fmri_kit.event_isc.contrast import parcels_to_nifti
from statsmodels.stats.multitest import fdrcorrection

## 1. Configuration

In [ ]:
ROOT           = Path('/path/to/your/project/root')  # CHANGE THIS
BEHAVIORAL_CSV = ROOT / 'behavioral_analyses/data/250226/merged_behavioral_bids.csv'

NPZ_DIR    = ROOT / 'data/derivatives/postbypost/parcel_patterns'
ATLAS_NII  = ROOT / 'data/atlases/Schaefer2018_tf_2mm_400Parcels7Networks_plus_TianS3.dseg.nii.gz'
LABELS_TSV = ROOT / 'data/atlases/Schaefer2018_400Parcels7Networks_plus_TianS3_labels.tsv'

OUTPUT_DIR = ROOT / 'data/derivatives/parcel_ispc/leftwing'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TYPES = ['AntiLeft', 'AntiRight', 'ProLeft', 'ProRight']
FDR_Q     = 0.05
TOP_N     = 20   # parcels to show in bar chart

## 2. Load Left-Wing Subjects

In [ ]:
beh_df   = pd.read_csv(BEHAVIORAL_CSV)
beh_df   = beh_df.drop_duplicates(subset='bids_id', keep='first')
subjects = sorted(beh_df[beh_df['political_group'] == 'left']['bids_id'].tolist())

print(f'Left-wing subjects ({len(subjects)}):')
print(subjects)

## 3. Check Extraction Status

NPZ files are produced by `scripts/extract_parcel_patterns.py`.  
Run the cell below to check how many files exist, or execute the script if needed:

```bash
cd /path/to/your/project/root  # CHANGE THIS
python yy-fMRI-kit/scripts/extract_parcel_patterns.py

```

In [ ]:
npzs = sorted(NPZ_DIR.glob('**/*_desc-parcel_patterns.npz'))
print(f'NPZ files found: {len(npzs)}  (expected {len(subjects) * len(RUN_TYPES)} = '
      f'{len(subjects)} subjects x {len(RUN_TYPES)} conditions)')

if npzs:
    sample = np.load(npzs[0], allow_pickle=True)
    print(f'\nSample: {npzs[0].relative_to(NPZ_DIR)}')
    print(f'  data shape          : {sample["data"].shape}  (n_posts, n_brain_voxels)')
    print(f'  n_parcels in labels : {len(sample["parcel_ids"])}')
else:
    print('\nNo NPZ files found — run extract_parcel_patterns.py first.')

## 4. Run Per-Parcel Pattern ISC — All Conditions

In [ ]:
results = run_parcel_isc(
    npz_dir   = NPZ_DIR,
    subjects  = subjects,
    run_types = RUN_TYPES,
)

## 5. Results Table + FDR Correction

FDR correction (Benjamini-Hochberg) is applied across parcels within each condition.
Note: p-values require a permutation test (deferred).  The table below shows ISC values without significance — update once permutation is run.

In [ ]:
df = results_to_dataframe(results)

# Placeholder p-value columns (fill once permutation test is implemented)
df['p_perm'] = np.nan
df['p_fdr']  = np.nan

# Summary per condition: top 10 by ISC
for cond in RUN_TYPES:
    sub = df[df['condition'] == cond].sort_values('isc_mean', ascending=False)
    print(f'\n── {cond} ──  (mean ISC across parcels: {sub["isc_mean"].mean():.4f})')
    print(sub[['parcel_name','isc_mean']].head(10).to_string(index=False))

## 6. Bar Chart — Top Parcels per Condition

Shows the `TOP_N` parcels with the highest mean ISC for each condition.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10), sharey=False)
axes_flat = axes.flatten()
colors    = ['#E07B54', '#5B8DB8', '#6BAE75', '#A97FC4']

for ax, cond, color in zip(axes_flat, RUN_TYPES, colors):
    sub = (df[df['condition'] == cond]
           .sort_values('isc_mean', ascending=False)
           .head(TOP_N))
    ax.barh(sub['parcel_name'][::-1], sub['isc_mean'][::-1],
            color=color, edgecolor='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.8, linestyle='--')
    ax.set_title(f'{cond}  (top {TOP_N} parcels)', fontweight='bold')
    ax.set_xlabel('Mean ISC (Pearson r)')
    ax.tick_params(axis='y', labelsize=7)
    ax.spines[['top','right']].set_visible(False)

fig.suptitle(
    'Per-Parcel Voxel-Pattern ISC — Left-Wing Subjects',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
fig_path = OUTPUT_DIR / 'parcel_ispc_barplot_top20.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Saved → {fig_path}')
plt.show()

## 7. ISC Distribution Across Parcels

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
colors    = ['#E07B54', '#5B8DB8', '#6BAE75', '#A97FC4']

for ax, cond, color in zip(axes, RUN_TYPES, colors):
    vals = df[df['condition'] == cond]['isc_mean'].dropna()
    ax.hist(vals, bins=40, color=color, edgecolor='k', linewidth=0.3, alpha=0.85)
    ax.axvline(vals.mean(), color='k', linewidth=1.5, linestyle='--',
               label=f'mean={vals.mean():.3f}')
    ax.set_title(cond, fontweight='bold')
    ax.set_xlabel('ISC (Pearson r)')
    ax.legend(fontsize=8)
    ax.spines[['top','right']].set_visible(False)
axes[0].set_ylabel('Number of parcels')

fig.suptitle('Distribution of per-parcel ISC across 432 parcels', fontsize=12)
plt.tight_layout()
dist_path = OUTPUT_DIR / 'parcel_ispc_distribution.png'
fig.savefig(dist_path, dpi=150, bbox_inches='tight')
print(f'Saved → {dist_path}')
plt.show()

## 8. Build ISC NIfTI Maps

Paint the mean ISC value for each parcel into its voxels using `parcels_to_nifti`
from `contrast.py`.  One NIfTI per condition.

In [ ]:
NIFTI_DIR = OUTPUT_DIR / 'nifti_maps'
NIFTI_DIR.mkdir(exist_ok=True)

nifti_paths = {}

for cond in RUN_TYPES:
    if cond not in results:
        print(f'{cond}: no results — skipping')
        continue

    r            = results[cond]
    parcel_names = [str(n) for n in r['parcel_names']]
    isc_values   = r['isc_mean'].astype(float)   # (n_parcels,)

    out_path = NIFTI_DIR / f'parcel_isc_{cond}.nii.gz'
    parcels_to_nifti(
        values       = isc_values,
        parcel_names = parcel_names,
        atlas_nii    = ATLAS_NII,
        labels_tsv   = LABELS_TSV,
        output_path  = out_path,
    )
    nifti_paths[cond] = out_path
    print(f'{cond}: saved → {out_path.name}')

print(f'\n{len(nifti_paths)} NIfTI maps built')

## 9. Yabplot — Static Surface Brain Maps

Project each condition's ISC NIfTI onto the fsLR-32k midthickness surface.  
Colour scale is shared across all conditions (symmetric ± max |ISC|).

In [ ]:
import yabplot as yab
import yabplot.data as ydata

_lh_surf, _rh_surf = ydata.get_surface_paths('midthickness', 'bmesh')

ALL_VIEWS = [
    'left_lateral', 'left_medial', 'right_lateral', 'right_medial',
    'superior', 'inferior', 'anterior', 'posterior',
]

# Shared symmetric colour scale driven by max |ISC| across all conditions
all_isc = [float(v) for cond in results for v in results[cond]['isc_mean'] if not np.isnan(v)]
VMAX    = max(abs(v) for v in all_isc)
VMAX    = max(VMAX, 0.01)
VMINMAX = [-VMAX, VMAX]
print(f'Shared colour scale: [{-VMAX:.4f}, {VMAX:.4f}]')

STATIC_DIR = OUTPUT_DIR / 'brain_maps_static'
STATIC_DIR.mkdir(exist_ok=True)

for cond, nii_path in nifti_paths.items():
    lh_data, rh_data = yab.project_vol2surf(str(nii_path), interpolation='nearest')
    lh_mesh, rh_mesh = yab.load_vertexwise_mesh(_lh_surf, _rh_surf, lh_data, rh_data)
    yab.plot_vertexwise(
        lh_mesh, rh_mesh,
        views        = ALL_VIEWS,
        cmap         = 'Reds',
        vminmax      = [0.1, 0.7],
        figsize      = (1600, 800),
        display_type = 'static',
        export_path  = str(STATIC_DIR / f'parcel_isc_{cond}.png'),
    )
    print(f'{cond}: saved → parcel_isc_{cond}.png')

## 10. Nilearn — Interactive HTML Brain Maps

Generates an interactive HTML viewer per condition.  Parcels with ISC below `EPS` are transparent (background).

In [ ]:
import nibabel as nib
from nilearn import plotting
from IPython.display import IFrame, display

HTML_DIR = OUTPUT_DIR / 'brain_maps_html'
HTML_DIR.mkdir(exist_ok=True)

EPS = 1e-6

for cond, nii_path in nifti_paths.items():
    img = nib.load(str(nii_path))
    html_view = plotting.view_img(
        img,
        bg_img         = 'MNI152',
        cmap           = 'Reds',
        threshold      = 0.2,
        vmin           = 0.2,
        vmax           = 0.7,
        title          = f'Per-Parcel Pattern ISC — {cond} (left-wing)',
        symmetric_cmap = False,
    )
    out_path = HTML_DIR / f'parcel_isc_{cond}.html'
    html_view.save_as_html(str(out_path))
    print(f'{cond}: saved → {out_path.name}')

# Display first condition inline
display(IFrame(str(HTML_DIR / f'parcel_isc_{RUN_TYPES[0]}.html'), width='100%', height=500))

## 11. Save Results

In [ ]:
# Summary table (one row per condition × parcel)
out_csv = OUTPUT_DIR / 'parcel_ispc_leftwing_summary.csv'
df.to_csv(out_csv, index=False)
print(f'Summary saved → {out_csv}')

# Per-subject ISC (one file per condition)
for cond in RUN_TYPES:
    if cond not in results:
        continue
    r    = results[cond]
    rows = []
    for p_idx, (pid, pname) in enumerate(zip(r['parcel_ids'], r['parcel_names'])):
        for sub, val in zip(r['subjects'], r['isc_subj'][:, p_idx]):
            rows.append({'condition': cond, 'parcel_id': int(pid),
                         'parcel_name': str(pname), 'subject': sub,
                         'isc_r': round(float(val), 6)})
    subj_df  = pd.DataFrame(rows)
    subj_csv = OUTPUT_DIR / f'parcel_ispc_{cond}_persubject.csv'
    subj_df.to_csv(subj_csv, index=False)
    print(f'Per-subject saved → {subj_csv.name}')

df

---

## Approach B: Post-wise ISC (Chen et al. 2017)

In Approach A (sections above), each subject's 18 post-patterns are averaged into
one condition-level mean *before* the inter-subject correlation is computed.

In **Approach B**, a separate leave-one-out ISC is computed for every post
(correlating each subject's voxel pattern for that post against the group mean
for that post), and the resulting per-post ISC values are averaged across posts.

This matches the structure in Chen et al. (2017): pattern vectors are correlated
per scene (post) between subjects, and scene-level correlations are averaged.

| | Approach A | Approach B |
|---|---|---|
| Step 1 | Average 18 posts → one mean pattern | Keep 18 patterns separately |
| Step 2 | LOO Pearson r across voxels | LOO Pearson r per post, then average |
| SNR | Higher (averaging removes noise) | Lower (per-post noise retained) |
| Post-specificity | None (condition-level) | Preserved |


In [ ]:
from yy_fmri_kit.event_isc.parcel_pattern_similarity import (
    run_parcel_isc_postwise,
    results_to_dataframe,
)

results_b = run_parcel_isc_postwise(
    npz_dir    = NPZ_DIR,
    subjects   = subjects,
    run_types  = RUN_TYPES,
    min_voxels = 5,
)


## B.1 Results Table

In [ ]:
df_b = results_to_dataframe(results_b)
df_b['approach'] = 'B_postwise'

print("Approach B — post-wise ISC summary (mean ISC across parcels per condition)")
summary = df_b.groupby('condition')['isc_mean'].agg(['mean', 'max', 'min']).round(4)
print(summary)
print()
print(f"Top-5 parcels per condition:")
for cond in RUN_TYPES:
    if cond not in results_b:
        continue
    top = df_b[df_b['condition'] == cond].nlargest(5, 'isc_mean')[['parcel_name','isc_mean']]
    print(f"  {cond}:")
    print(top.to_string(index=False))
    print()


## B.2 ISC Distribution Across Parcels

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
colors = ['#E07B54', '#5B8DB8', '#6AA96B', '#9B6BB5']

for ax, (cond, color) in zip(axes, zip(RUN_TYPES, colors)):
    if cond not in results_b:
        ax.set_visible(False)
        continue
    vals = results_b[cond]['isc_mean']
    valid = vals[~np.isnan(vals)]
    ax.hist(valid, bins=40, color=color, alpha=0.75, edgecolor='white')
    ax.axvline(valid.mean(), color='black', lw=1.5, ls='--',
               label=f'mean={valid.mean():.3f}')
    ax.set_title(cond, fontsize=12)
    ax.set_xlabel('ISC (r)', fontsize=10)
    ax.legend(fontsize=9)

axes[0].set_ylabel('Parcel count', fontsize=10)
fig.suptitle('Approach B: Post-wise ISC — Distribution Across Parcels', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'parcel_isc_B_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


## B.3 Build ISC NIfTI Maps

In [ ]:
from yy_fmri_kit.event_isc.contrast import parcels_to_nifti

NIFTI_DIR_B = OUTPUT_DIR / 'nifti_maps_B'
NIFTI_DIR_B.mkdir(exist_ok=True)

nifti_paths_b = {}
for cond, r in results_b.items():
    out_path = NIFTI_DIR_B / f'parcel_isc_B_{cond}.nii.gz'
    nii = parcels_to_nifti(
        r['isc_mean'], r['parcel_names'], ATLAS_NII, LABELS_TSV, out_path
    )
    nifti_paths_b[cond] = out_path
    valid = r['isc_mean'][~np.isnan(r['isc_mean'])]
    print(f'  {cond}: saved {out_path.name}  '
          f'(range [{valid.min():.4f}, {valid.max():.4f}])')


In [ ]:
import yabplot as yab
import yabplot.data as ydata

# Reuse _lh_surf, _rh_surf and ALL_VIEWS already defined in the Approach A yabplot cell above

B_VMAX = max(
    float(np.nanmax(np.abs(results_b[c]['isc_mean'])))
    for c in RUN_TYPES if c in results_b
)
B_VMAX    = max(B_VMAX, 0.01)
VMINMAX_B = [-B_VMAX, B_VMAX]
print(f'Approach B colour scale: [{-B_VMAX:.4f}, {B_VMAX:.4f}]')

STATIC_DIR_B = OUTPUT_DIR / 'brain_maps_static_B'
STATIC_DIR_B.mkdir(exist_ok=True)

for cond, nii_path in nifti_paths_b.items():
    lh_data, rh_data = yab.project_vol2surf(str(nii_path), interpolation='nearest')
    lh_mesh, rh_mesh = yab.load_vertexwise_mesh(_lh_surf, _rh_surf, lh_data, rh_data)
    yab.plot_vertexwise(
        lh_mesh, rh_mesh,
        views        = ALL_VIEWS,
        cmap         = 'Reds',
        vminmax      = [0, 0.4],
        figsize      = (1600, 800),
        display_type = 'static',
        export_path  = str(STATIC_DIR_B / f'parcel_isc_B_{cond}.png'),
    )
    print(f'{cond}: saved → parcel_isc_B_{cond}.png')


In [ ]:
from nilearn import plotting
from IPython.display import IFrame, display

HTML_DIR_B = OUTPUT_DIR / 'brain_maps_html_B_reds'
HTML_DIR_B.mkdir(exist_ok=True)
EPS = 1e-6

for cond, nii_path in nifti_paths_b.items():
    img = nib.load(str(nii_path))
    html_view = plotting.view_img(
        img,
        bg_img         = 'MNI152',
        cmap           = 'Reds',
        threshold      = 0.1,
        vmin           = 0.1,
        vmax           = 0.4,
        title          = f'Per-Parcel Pattern ISC (Approach B) — {cond} (left-wing)',
        symmetric_cmap = False,
    )
    out_path = HTML_DIR_B / f'parcel_isc_B_{cond}.html'
    html_view.save_as_html(str(out_path))
    print(f'{cond}: saved → {out_path.name}')

display(IFrame(str(HTML_DIR_B / f'parcel_isc_B_{RUN_TYPES[0]}.html'), width=900, height=500))



---

## Approach A vs B: Comparison

Scatter plot of Approach A (mean-pattern ISC) vs Approach B (post-wise ISC) for
every parcel × condition.  Points on the diagonal indicate perfect agreement.

- A systematic upward shift of A relative to B is expected (averaging before
  correlating inflates r by reducing noise).
- Similar rank ordering across parcels would confirm that both methods identify
  the same brain regions as showing the strongest inter-subject pattern similarity.


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
colors = ['#E07B54', '#5B8DB8', '#6AA96B', '#9B6BB5']

for ax, (cond, color) in zip(axes, zip(RUN_TYPES, colors)):
    if cond not in results or cond not in results_b:
        ax.set_visible(False)
        continue
    a_vals = results[cond]['isc_mean']
    b_vals = results_b[cond]['isc_mean']
    valid  = ~(np.isnan(a_vals) | np.isnan(b_vals))
    ax.scatter(a_vals[valid], b_vals[valid],
               alpha=0.35, s=7, color=color, linewidths=0)
    lim = max(np.abs(a_vals[valid]).max(), np.abs(b_vals[valid]).max()) * 1.1
    ax.plot([-lim, lim], [-lim, lim], 'k--', lw=0.8, alpha=0.6, label='identity')
    ax.axhline(0, color='gray', lw=0.5, alpha=0.4)
    ax.axvline(0, color='gray', lw=0.5, alpha=0.4)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_xlabel('Approach A (mean-pattern ISC)', fontsize=9)
    ax.set_ylabel('Approach B (post-wise ISC)', fontsize=9)
    ax.set_title(cond, fontsize=11)
    r = np.corrcoef(a_vals[valid], b_vals[valid])[0, 1]
    ax.text(0.05, 0.92, f'r = {r:.3f}', transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

fig.suptitle('Approach A vs B: ISC per Parcel (all conditions)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'parcel_isc_AB_scatter.png', dpi=150, bbox_inches='tight')
plt.show()


## Save Approach B Results

In [ ]:
out_b = OUTPUT_DIR / 'parcel_isc_B_summary.csv'
df_b.to_csv(out_b, index=False)
print(f'Saved {out_b.name}')

for cond, r in results_b.items():
    subj_df = pd.DataFrame(
        r['isc_subj'],
        columns=r['parcel_names'],
        index=r['subjects'],
    )
    out_subj = OUTPUT_DIR / f'parcel_isc_B_{cond}_persubject.csv'
    subj_df.to_csv(out_subj)
    print(f'  Saved {out_subj.name}')
